Setting the python path to import code from app/src.

The extrapath is configured at .vscode/settings.json to avoid IDE errors.

In [ ]:
import sys
from pathlib import Path


def find_project_root(marker="pyproject.toml") -> Path:
    p = Path.cwd().resolve()
    for parent in [p, *p.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"{marker} not found starting from {p}")


root_dir = find_project_root()
sys.path.append(str(root_dir / "app" / "src"))
root_dir


Imports

In [ ]:
from pathlib import Path

from application.use_cases.boilerplate_removal import BOILERPLATE_REMOVAL_VERSION
from application.use_cases.clause_classification import classify_and_enrich_clauses
from application.use_cases.clause_segmentation import (
    CLAUSE_SEGMENTATION_VERSION,
    segment_document,
)
from domain.clause_tree import Clause, ClauseTree
from infrastructure.parsing.boilerplate_caching import (
    boilerplate_cache_path,
    compute_boilerplate_cache_key,
)
from infrastructure.parsing.caching import read_cache
from infrastructure.parsing.clause_schema import (
    SCHEMA_VERSION,
    ParsedClauseRecord,
    flatten_typed_clause,
)
from infrastructure.parsing.clause_tree_caching import (
    clause_tree_cache_path,
    compute_clause_tree_cache_key,
    read_clause_tree_cache,
)
from infrastructure.parsing.corpus_artifact import (
    BuildManifest,
    read_parsed_clauses_jsonl,
    read_parsed_clauses_parquet,
    utc_now,
    write_build_manifest,
    write_parsed_clauses_jsonl,
    write_parsed_clauses_parquet,
)
from infrastructure.parsing.manifest import read_manifest
from infrastructure.parsing.null_classifier import NullClauseClassifier
from infrastructure.parsing.rules_loader import load_classification_rules

MANIFEST_PATH = root_dir / "data" / "policies" / "manifest.csv"
RULES_PATH = root_dir / "data" / "parsing" / "clause_type_mapping.csv"


path and clause_id deterministic

In [ ]:
from domain.extracted_text import ExtractedDocument, ExtractedPage, ExtractedSpan


def _span(page_number, line_id, text, *, bold):
    return ExtractedSpan(
        document_id="scratch",
        page_number=page_number,
        line_id=line_id,
        order=line_id,
        bbox=(50.0, 100.0 + line_id * 15.0, 50.0 + len(text) * 6.0, 110.0 + line_id * 15.0),
        font_size=11.0,
        font_name="Test-BoldMT" if bold else "TestMT",
        text=text,
    )


def _document(lines: list[tuple[str, bool]]) -> ExtractedDocument:
    spans = [_span(1, i, text, bold=bold) for i, (text, bold) in enumerate(lines)]
    page = ExtractedPage(page_number=1, spans=tuple(spans), char_count=sum(len(t) for t, _ in lines))
    return ExtractedDocument(document_id="scratch", filename="scratch.pdf", pages=(page,), extractor_version="v1")


baseline_doc = _document([
    ("1. OBJETO DO SEGURO", True),
    ("Corpo do objeto.", False),
    ("2. COBERTURAS", True),
    ("2.1 Cobertura Basica", True),
    ("Texto da cobertura basica.", False),
])

with_insertion_doc = _document([
    ("1. OBJETO DO SEGURO", True),
    ("Corpo do objeto.", False),
    ("1.1 Definicoes Extras", True),
    ("Texto das definicoes extras.", False),
    ("2. COBERTURAS", True),
    ("2.1 Cobertura Basica", True),
    ("Texto da cobertura basica.", False),
])

tree_a = segment_document(baseline_doc)
tree_b = segment_document(with_insertion_doc)

for clause in tree_a.all_clauses:
    print(clause.numbering_label, "-", clause.path, "-", clause.clause_id)

print()
print("clause_count A/B:", tree_a.report.clause_count, tree_b.report.clause_count)


def find(tree: ClauseTree, numbering_label: str) -> Clause:
    return next(c for c in tree.all_clauses if c.numbering_label == numbering_label)


assert find(tree_a, "2").clause_id == find(tree_b, "2").clause_id
assert find(tree_a, "2.1").clause_id == find(tree_b, "2.1").clause_id
print("clause_id de '2' e '2.1' inalterado apesar da cláusula extra inserida em '1'.")

tree_a_rerun = segment_document(baseline_doc)
assert [c.clause_id for c in tree_a.all_clauses] == [c.clause_id for c in tree_a_rerun.all_clauses]



Using a real doc from corpus

In [ ]:
manifest_records = read_manifest(MANIFEST_PATH)
document_id = "10"
entry = next(r for r in manifest_records if r["id"] == document_id)
print(entry["filename"], entry["product_line"], entry["insurer"])

boilerplate_key = compute_boilerplate_cache_key(BOILERPLATE_REMOVAL_VERSION)
boilerplate_path = boilerplate_cache_path(document_id, boilerplate_key)
document = read_cache("../../" + str(boilerplate_path))

tree = segment_document(document)
print("clauses:", tree.report.clause_count, "max_depth:", tree.report.max_depth)
print("orphan_ratio:", round(tree.report.orphan_ratio, 4))

for clause in tree.all_clauses[:15]:
    print(f"{clause.path:20s} depth={clause.depth} bundle={clause.bundle_section!r} title={clause.title[:60]!r}")


Reading from cache

In [ ]:
clause_tree_key = compute_clause_tree_cache_key(CLAUSE_SEGMENTATION_VERSION)
clause_tree_path = clause_tree_cache_path(document_id, clause_tree_key)
cached_tree = read_clause_tree_cache("../../" + str(clause_tree_path))

with_bundle = [c for c in cached_tree.all_clauses if c.bundle_section]
print(f"{len(with_bundle)}/{len(cached_tree.all_clauses)} cláusulas com bundle_section definido")
with_bundle[0].bundle_section, with_bundle[0].bundle_confidence


Classification

In [ ]:
rules = load_classification_rules(RULES_PATH)
classifier = NullClauseClassifier()

typed_clauses = classify_and_enrich_clauses(cached_tree, manifest_records, rules, classifier)

source = "ocr" if entry["extraction_mode"] == "ocr_required" else "text"
records = [flatten_typed_clause(typed, source=source) for typed in typed_clauses]

print(f"{len(records)} generated, schema_version={records[0].schema_version}")
records[0]


In [ ]:
import collections

print("clause_type:", collections.Counter(r.clause_type for r in records))
print("type_source:", collections.Counter(r.type_source for r in records))


Let's see the provenance validation failing

In [ ]:
from dataclasses import replace

from pydantic import ValidationError

typed_sample = typed_clauses[0]
broken_provenance = replace(typed_sample.provenance, insurer="")
broken_typed = replace(typed_sample, provenance=broken_provenance)

try:
    flatten_typed_clause(broken_typed, source=source)
except ValidationError as exc:
    print("Failed:")
    print(exc)


round-trip parquet/JSONL and build manifest

In [ ]:
scratch_dir = root_dir / "notebooks" / "scratch" / "_m1_07_scratch_output"
scratch_dir.mkdir(parents=True, exist_ok=True)

parquet_path = scratch_dir / "parsed_clauses.parquet"
jsonl_path = scratch_dir / "parsed_clauses.jsonl"

write_parsed_clauses_parquet(records, parquet_path)
write_parsed_clauses_jsonl(records, jsonl_path)

restored_parquet = read_parsed_clauses_parquet(parquet_path)
restored_jsonl = read_parsed_clauses_jsonl(jsonl_path)

assert restored_parquet == records
assert restored_jsonl == records
print("Round-trip OK:", len(records), "registers")

manifest = BuildManifest(
    schema_version=SCHEMA_VERSION,
    clause_segmentation_version=CLAUSE_SEGMENTATION_VERSION,
    boilerplate_removal_version=BOILERPLATE_REMOVAL_VERSION,
    llm_classification_enabled=False,
    built_at_utc=utc_now(),
    clause_counts_by_document={document_id: len(records)},
    total_clause_count=len(records),
)
write_build_manifest(manifest, scratch_dir / "manifest.json")
manifest


Now I'm going to check the real artifact built by make parse

In [ ]:
real_parquet = root_dir / "build" / "parsed_clauses.parquet"

if real_parquet.exists():
    all_records = read_parsed_clauses_parquet(real_parquet)
    print(f"{len(all_records)} clauses in {len(set(r.document_id for r in all_records))} docs")
    print(collections.Counter(r.source for r in all_records))
    print(collections.Counter(r.clause_type for r in all_records))
else:
    print("build/parsed_clauses.parquet does not exist yet. Run `make parse`.")


Let's clean up things

In [ ]:
import shutil

shutil.rmtree(scratch_dir, ignore_errors=True)